<a href="https://colab.research.google.com/github/mithun30052001/capstone-project/blob/main/Part3-Advanced-Modeling/Advanced_Modeling_and_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
import joblib

# Load data
print("Loading cleaned dataset...")
df = pd.read_csv('cleaned_data.csv', low_memory=False)

# Targets
y_reg = df['days_to_forward']
y_clf = (y_reg > y_reg.quantile(0.75)).astype(int)

# Drop raw date/text/ID columns to prevent leakage
cols_to_drop = [
    'days_to_forward', 'date_received', 'date_sent',
    'Date received', 'Date sent to company',
    'Consumer complaint narrative', 'Company public response',
    'ZIP code', 'Complaint ID'
]
X_raw = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# Preprocess categorical columns (same as Part 2)
speed_mapping = {
    'Web': 1,
    'Phone': 2,
    'Referral': 2,
    'Fax': 3,
    'Postal mail': 3
}
if 'Submitted via' in X_raw.columns:
    X_raw['Submitted via_encoded'] = X_raw['Submitted via'].map(speed_mapping).fillna(2)
    X_raw = X_raw.drop(columns=['Submitted via'])

cat_cols = X_raw.select_dtypes(include=['object', 'category']).columns.tolist()
X_encoded = pd.get_dummies(X_raw, columns=cat_cols, drop_first=True)
X_encoded = X_encoded.fillna(0)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_clf, test_size=0.2, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data successfully split and scaled.")

Loading cleaned dataset...
Data successfully split and scaled.


In [ ]:
# Task 1: Unconstrained Decision Tree
dt_unconstrained = DecisionTreeClassifier(random_state=42)
dt_unconstrained.fit(X_train_scaled, y_train)

train_acc_un = accuracy_score(y_train, dt_unconstrained.predict(X_train_scaled))
test_acc_un = accuracy_score(y_test, dt_unconstrained.predict(X_test_scaled))

print("--- Task 1: Unconstrained Decision Tree ---")
print(f"Train Accuracy: {train_acc_un:.4f}")
print(f"Test Accuracy:  {test_acc_un:.4f}")

# Task 2: Controlled Decision Tree
dt_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_controlled.fit(X_train_scaled, y_train)

train_acc_con = accuracy_score(y_train, dt_controlled.predict(X_train_scaled))
test_acc_con = accuracy_score(y_test, dt_controlled.predict(X_test_scaled))

print("\n--- Task 2: Controlled Decision Tree (max_depth=5, min_samples_split=20) ---")
print(f"Train Accuracy: {train_acc_con:.4f}")
print(f"Test Accuracy:  {test_acc_con:.4f}")

# Task 3: Gini vs Entropy
dt_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_gini.fit(X_train_scaled, y_train)
test_acc_gini = accuracy_score(y_test, dt_gini.predict(X_test_scaled))

dt_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42)
dt_entropy.fit(X_train_scaled, y_train)
test_acc_entropy = accuracy_score(y_test, dt_entropy.predict(X_test_scaled))

print("\n--- Task 3: Split Criteria Comparison (max_depth=5) ---")
print(f"Gini Test Accuracy:    {test_acc_gini:.4f}")
print(f"Entropy Test Accuracy: {test_acc_entropy:.4f}")

--- Task 1: Unconstrained Decision Tree ---
Train Accuracy: 0.9989
Test Accuracy:  0.9889

--- Task 2: Controlled Decision Tree (max_depth=5, min_samples_split=20) ---
Train Accuracy: 0.9926
Test Accuracy:  0.9922

--- Task 3: Split Criteria Comparison (max_depth=5) ---
Gini Test Accuracy:    0.9922
Entropy Test Accuracy: 0.9920


In [11]:
# Task 4: Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled, y_train)

rf_train_acc = accuracy_score(y_train, rf_model.predict(X_train_scaled))
rf_test_acc = accuracy_score(y_test, rf_model.predict(X_test_scaled))
rf_test_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test_scaled)[:, 1])

print("--- Task 4: Random Forest (n_estimators=100, max_depth=10) ---")
print(f"Train Accuracy: {rf_train_acc:.4f}")
print(f"Test Accuracy:  {rf_test_acc:.4f}")
print(f"Test ROC-AUC:   {rf_test_auc:.4f}")

# Extract top 5 features
importances = rf_model.feature_importances_
feat_importance = pd.DataFrame({
    'Feature': X_encoded.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\nTop 5 Features by Random Forest Importance:")
display(feat_importance.head(5))

# Task 4a: Gradient Boosting Classifier
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train_scaled, y_train)

gb_train_acc = accuracy_score(y_train, gb_model.predict(X_train_scaled))
gb_test_acc = accuracy_score(y_test, gb_model.predict(X_test_scaled))
gb_test_auc = roc_auc_score(y_test, gb_model.predict_proba(X_test_scaled)[:, 1])

print("\n--- Task 4a: Gradient Boosting ---")
print(f"Train Accuracy: {gb_train_acc:.4f}")
print(f"Test Accuracy:  {gb_test_acc:.4f}")
print(f"Test ROC-AUC:   {gb_test_auc:.4f}")

# Task 4b: Feature Ablation Study
# Identify 5 lowest importance features
lowest_5_feats = feat_importance.tail(5)['Feature'].tolist()
print(f"\n--- Task 4b: Feature Ablation Study ---")
print(f"5 Lowest Importance Features to Remove: {lowest_5_feats}")

# Drop from scaled sets
X_train_reduced = pd.DataFrame(X_train_scaled, columns=X_encoded.columns).drop(columns=lowest_5_feats)
X_test_reduced = pd.DataFrame(X_test_scaled, columns=X_encoded.columns).drop(columns=lowest_5_feats)

# Train reduced RF
rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_reduced, y_train)
rf_reduced_auc = roc_auc_score(y_test, rf_reduced.predict_proba(X_test_reduced)[:, 1])

print(f"Full Model Test AUC:    {rf_test_auc:.4f}")
print(f"Reduced Model Test AUC: {rf_reduced_auc:.4f}")
print(f"AUC Difference:         {rf_test_auc - rf_reduced_auc:.4f}")


--- Task 4: Random Forest (n_estimators=100, max_depth=10) ---
Train Accuracy: 0.9920
Test Accuracy:  0.9922
Test ROC-AUC:   0.8908

Top 5 Features by Random Forest Importance:


,Feature,Importance
1051,Company_SYNCHRONY FINANCIAL,0.044302
21,Sub-product_Credit reporting,0.042186
1287,Timely response?_Yes,0.042169
0,narrative_length,0.041831
1066,Company_Servicer under contract with Federal S...,0.041493



--- Task 4a: Gradient Boosting ---
Train Accuracy: 0.9936
Test Accuracy:  0.9923
Test ROC-AUC:   0.8590

--- Task 4b: Feature Ablation Study ---
5 Lowest Importance Features to Remove: ['Company_MCMC Auto LTD', 'Company_M. Richard Epps, P.C.', 'Company_Medical Data Systems, Inc.', 'Company_BLUE WATER CREDIT, LLC', 'Company_Capital Management Services, LP']
Full Model Test AUC:    0.8908
Reduced Model Test AUC: 0.8801
AUC Difference:         0.0106


In [12]:
from sklearn.linear_model import LogisticRegression

# Initialize Logistic Regression from Part 2 (C=1.0)
log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0)

# Models dictionary
models = {
    'Logistic Regression': log_reg,
    'Controlled DT': dt_controlled,
    'Random Forest': rf_model
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("--- Task 5: 5-Fold Cross-Validation Comparison (ROC-AUC) ---")
cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:20} -> Mean AUC: {scores.mean():.4f} | Std AUC: {scores.std():.4f}")

--- Task 5: 5-Fold Cross-Validation Comparison (ROC-AUC) ---
Logistic Regression  -> Mean AUC: 0.6676 | Std AUC: 0.0610
Controlled DT        -> Mean AUC: 0.7435 | Std AUC: 0.0356
Random Forest        -> Mean AUC: 0.8971 | Std AUC: 0.0064


In [14]:
# Task 6: Hyperparameter Tuning with GridSearchCV Pipeline (Optimized for Speed)
from sklearn.model_selection import StratifiedKFold

# 1. Define a smaller, faster parameter grid (removed None and 200 estimators to prevent infinite tree growth)
param_grid = {
    'randomforestclassifier__n_estimators': [50, 100],
    'randomforestclassifier__max_depth': [5, 10],
    'randomforestclassifier__min_samples_leaf': [5]
}

# 2. Build Pipeline
pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

# 3. Subsample 20% of training data for fast grid search (stratified to maintain class balance)
X_train_sub, _, y_train_sub, _ = train_test_split(
    X_train, y_train, test_size=0.8, stratify=y_train, random_state=42
)

print(f"--- Task 6: Running GridSearchCV on 20% subset ({X_train_sub.shape[0]} rows)... ---")
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42), # Reduced CV folds to 3 for search speed
    scoring='roc_auc',
    n_jobs=-1
)
grid_search.fit(X_train_sub, y_train_sub)

print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Best Subset CV ROC-AUC Score: {grid_search.best_score_:.4f}")

# 4. CRITICAL STEP: Re-train the best estimator on 100% of the training data
print("\nRe-fitting the best pipeline on 100% of the training data...")
best_pipeline = grid_search.best_estimator_
best_pipeline.fit(X_train, y_train)
print("Fitting complete! You can now proceed to the Learning Curve and serialization tasks.")

--- Task 6: Running GridSearchCV on 20% subset (15991 rows)... ---
Best Parameters Found: {'randomforestclassifier__max_depth': 5, 'randomforestclassifier__min_samples_leaf': 5, 'randomforestclassifier__n_estimators': 100}
Best Subset CV ROC-AUC Score: 0.8684

Re-fitting the best pipeline on 100% of the training data...
Fitting complete! You can now proceed to the Learning Curve and serialization tasks.


In [15]:
# Task 7: Manual Learning Curve
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
curve_results = []

X_train_reset = X_train.reset_index(drop=True)
y_train_reset = y_train.reset_index(drop=True)

print("--- Task 7: Manual Learning Curve ---")
for f in fractions:
    subset_size = int(f * len(X_train_reset))
    X_subset = X_train_reset.iloc[:subset_size]
    y_subset = y_train_reset.iloc[:subset_size]

    # Fit pipeline
    best_pipeline.fit(X_subset, y_subset)

    # Predict probabilities
    train_probs = best_pipeline.predict_proba(X_subset)[:, 1]
    test_probs = best_pipeline.predict_proba(X_test)[:, 1]

    train_auc = roc_auc_score(y_subset, train_probs)
    test_auc = roc_auc_score(y_test, test_probs)

    curve_results.append({
        'Training Fraction': f,
        'Training AUC': train_auc,
        'Test AUC': test_auc
    })

curve_df = pd.DataFrame(curve_results)
display(curve_df)

--- Task 7: Manual Learning Curve ---


,Training Fraction,Training AUC,Test AUC
0,0.2,0.906722,0.869783
1,0.4,0.908358,0.877783
2,0.6,0.904148,0.881307
3,0.8,0.901619,0.879322
4,1.0,0.892398,0.880163


In [16]:
# Task 8: Serialize the best model
print("--- Task 8: Model Serialization ---")
joblib.dump(best_pipeline, 'best_model.pkl')
print("Model successfully saved to 'best_model.pkl'")

# Reload-and-predict block (runs test rows)
print("\nTesting Model Reloading...")
loaded_model = joblib.load('best_model.pkl')

# Extract two hand-crafted dummy rows matching the schema of X_test
sample_rows = X_test.iloc[:2]
predictions = loaded_model.predict(sample_rows)

print("Hand-crafted predictions:")
for i, pred in enumerate(predictions):
    print(f"Row {i+1} prediction: {pred} ({'Extreme Delay' if pred == 1 else 'Normal Speed'})")

--- Task 8: Model Serialization ---
Model successfully saved to 'best_model.pkl'

Testing Model Reloading...
Hand-crafted predictions:
Row 1 prediction: 0 (Normal Speed)
Row 2 prediction: 0 (Normal Speed)
